In [1]:
library(Seurat)
library(ggplot2)
library(dplyr)

# ========== 参数设置 ==========
prefix <- "bent"  # 修改此处即可更改所有输出前缀
rds_path <- "/data/work/RNA_AraSeed/rds/cot/cot_recluster.rds"
output_dir <- "/data/work/RNA_AraSeed/QC/cot"
col_cluster <- "seurat_clusters"  # 分群列名

# ========== 读取数据 ==========
scRNA <- readRDS(rds_path)

cat("\n========================================\n")
cat("Figure: Cluster-level Violin Plot\n")
cat("========================================\n")

# 提取metadata
meta <- scRNA@meta.data

# 设置cluster为因子，按数字排序（修复版）
# 先转为字符型，确保所有值都被正确处理
meta[[col_cluster]] <- as.character(meta[[col_cluster]])
cluster_nums <- sort(as.numeric(unique(meta[[col_cluster]])))
meta[[col_cluster]] <- factor(
  meta[[col_cluster]],
  levels = as.character(cluster_nums)
)

# 构建绘图数据
plot_df_cluster <- data.frame(
  Cluster = meta[[col_cluster]],
  UMI     = meta$nCount_RNA,
  Genes   = meta$nFeature_RNA
)

# 定义cluster颜色（根据实际的cluster数量自动生成）
n_clusters <- length(unique(plot_df_cluster$Cluster))
cluster_colors <- setNames(
  scales::hue_pal()(n_clusters),
  levels(plot_df_cluster$Cluster)
)

# Cluster Violin绘图函数
make_cluster_violin <- function(df, value_col, title_text,
                                 ylab_text = "Count (log10 scale)") {
  ggplot(df, aes(x = Cluster, y = .data[[value_col]], fill = Cluster)) +
    geom_violin(
      scale = "width",
      trim = TRUE,
      width = 0.9,
      color = "black",
      linewidth = 0.3,
      draw_quantiles = c(0.25, 0.5, 0.75),
      alpha = 0.8
    ) +
    scale_fill_manual(values = cluster_colors) +
    scale_y_log10() +
    labs(title = title_text, x = "Cluster", y = ylab_text) +
    theme_classic(base_size = 14) +
    theme(
      plot.title = element_text(hjust = 0.5, face = "bold", size = 16),
      axis.text.x = element_text(angle = 45, hjust = 1, vjust = 1, size = 10),
      axis.text.y = element_text(size = 10),
      axis.title.y = element_text(size = 13),
      legend.position = "none",
      panel.grid.major.y = element_line(color = "grey90", linewidth = 0.3),
      panel.grid.minor.y = element_line(color = "grey95", linewidth = 0.2)
    )
}

# 生成UMI和Genes子图
p_umi <- make_cluster_violin(
  plot_df_cluster, "UMI",
  paste0(prefix, " - UMI Count per Cluster")
)

p_genes <- make_cluster_violin(
  plot_df_cluster, "Genes",
  paste0(prefix, " - Gene Count per Cluster")
)

# 计算每个cluster的统计信息
cluster_stats <- plot_df_cluster %>%
  group_by(Cluster) %>%
  summarise(
    n_cells      = n(),
    median_UMI   = median(UMI),
    mean_UMI     = mean(UMI),
    median_Genes = median(Genes),
    mean_Genes   = mean(Genes),
    .groups = 'drop'
  )

cat("\nPer-cluster QC statistics:\n")
print(cluster_stats)

# 保存合并图
output_file <- file.path(output_dir, paste0(prefix, "_VlnPlot_QC_by_cluster.pdf"))
pdf(output_file, width = 14, height = 6)
print(cowplot::plot_grid(p_umi, p_genes, ncol = 2))
dev.off()

cat("\n✓ VlnPlot已保存到:", output_file, "\n")

Attaching SeuratObject


Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union





Figure: Cluster-level Violin Plot

Per-cluster QC statistics:
# A tibble: 22 × 6
   Cluster n_cells median_UMI mean_UMI median_Genes mean_Genes
   <fct>     <int>      <dbl>    <dbl>        <dbl>      <dbl>
 1 0          3265      2138     3197.        1125       1416.
 2 1          3115      1937     2731.        1055       1265.
 3 2          3038      2960.    5019.        1466       1838.
 4 3          2848      2469     4088.        1359       1793.
 5 4          2555      1810     2460.        1015       1199.
 6 5          2416      2269     3328.         976.      1244.
 7 6          2292      2413     3803.        1192.      1500.
 8 7          2254      3027     5276.        1284.      1702.
 9 8          1997      1784     2260.         980       1123.
10 9          1629      1698     2299.         865       1042.
# ℹ 12 more rows


pdf 
  2


✓ VlnPlot已保存到: /data/work/RNA_AraSeed/QC/cot/bent_VlnPlot_QC_by_cluster.pdf 
